# 🎯 HỆ THỐNG ĐÁNH GIÁ ĐA DẠNG MẪU N HẠT (IN-MEMORY DIVERSITY EVALUATOR)
### Đo Lường Độc Lập: Intra-Prompt Vendi Score (Effective Modes), Pairwise LPIPS & CLIP Distance

> **Mục đích**: Notebook chuyên dụng dùng để đánh giá định lượng độ đa dạng (Diversity Metrics) cho các thư mục kết quả sinh mẫu $N$ hạt trong `Target_samples/` trên Google Drive.

---

### 🌟 Ba Tầng Chỉ Số Đo Lường Đa Dạng:
1. **Intra-Prompt Vendi Score ($\overline{\text{VS}}$)**: Đo *"Số lượng mode độc lập thực tế"* trong $N$ hạt thông qua entropy Von Neumann của ma trận Gram CLIP ViT-B/32. *(Dải giá trị: $[1.0, N]$. $N=4$ giống hệt nhau $\implies 1.0$; $N=4$ độc lập $\implies 4.0$)*.
2. **Intra-Prompt Pairwise LPIPS ($\overline{\mathcal{D}}_{\text{LPIPS}}$)**: Đo khoảng cách cảm thụ trực quan trung bình giữa $\binom{N}{2}$ cặp hạt bằng AlexNet LPIPS (nhạy với bố cục, góc chụp, tư thế đối tượng).
3. **Pairwise CLIP Cosine Distance**: Đo khoảng cách ngữ nghĩa trung bình $1 - \cos(v_i, v_j)$.
4. **Đường Biên Pareto (Quality vs Diversity)**: Tự động đối chiếu điểm ImageReward từ `results.json` để vẽ đồ thị chứng minh RS-SMC vượt trội toàn diện (Pareto Dominance).

## 1. Kiểm Tra Phần Cứng GPU & Thiết Lập Môi Trường
Hệ thống đo lường có thể chạy trên bất kỳ GPU nào (T4, L4, A100, V100). Toàn bộ quá trình đánh giá 553 prompts chỉ mất **< 30 giây**!

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Gắn Kết Google Drive & Đồng Bộ Mã Nguồn
Tự động kết nối Google Drive và cập nhật mã nguồn mới nhất từ GitHub repository.

In [ ]:
import os, subprocess, sys
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)
WORKDIR = "/content/Diffusion-LiDAR-Sampling"

# 2. Clone hoặc Pull repository mới nhất
if not os.path.exists(WORKDIR):
    print(f"📥 Đang clone repository từ GitHub về {WORKDIR}...")
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {WORKDIR}
else:
    print(f"🔄 Đang đồng bộ cập nhật commit mới nhất tại {WORKDIR}...")
    %cd {WORKDIR}
    !git pull origin main

# Thêm WORKDIR vào sys.path
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

print("\n✅ Đồng bộ môi trường và Google Drive hoàn tất!")
print(f"📁 Thư mục Target_samples trên Drive: {DRIVE_DIR}/Target_samples")


## 3. Cài Đặt Thư Viện Đo Lường (LPIPS & Transformers)
Cài đặt gói `lpips` và xác nhận sẵn sàng các mô hình `AlexNet LPIPS` và `CLIP ViT-B/32`.

In [ ]:
# Cài đặt thư viện lpips nếu chưa có
try:
    import lpips
    print("✅ Thư viện lpips đã sẵn sàng!")
except ImportError:
    print("📦 Đang cài đặt thư viện lpips...")
    !pip install -q lpips
    import lpips
    print("✅ Cài đặt lpips thành công!")

import importlib
import compute_diversity_metrics
importlib.reload(compute_diversity_metrics)

from compute_diversity_metrics import (
    compute_vendi_score,
    compute_pairwise_clip_distance,
    compute_pairwise_lpips_batch,
    preload_run_to_ram,
    evaluate_multiple_folders,
    format_markdown_table
)
print("✅ Toàn bộ module đánh giá Diversity In-Memory đã nạp thành công!")


## 4. Cấu Hình Danh Sách Thư Mục Cần Chấm Điểm Diversity
Bạn chỉ cần **điền tên các thư mục kết quả** trong `Target_samples/` (cách nhau bởi dấu phẩy hoặc dòng mới). Có thể dùng ký tự đại diện `*` (wildcard) để tự động khớp.

In [ ]:
# @title ⚙️ Nhập Tên Thư Mục Cần Chấm Điểm & Thiết Lập Thông Số
import os, glob

# ==============================================================================
# 🎯 1. DANH SÁCH THƯ MỤC CẦN SO SÁNH ĐỐI KHÁNG
# ==============================================================================
# Nhập các thư mục phân tách bằng dấu phẩy (hoặc dùng dấu wildcard *)
# Ví dụ so sánh đối đầu giữa Vanilla SMC và RS-SMC:
TARGET_FOLDERS_INPUT = "SMC_SD15_N4_LiDAR_*, RSSMC_SD15_N4_sig0.25_M8_LiDAR_*" # @param {type:"string"}

# ==============================================================================
# ⚙️ 2. THIẾT LẬP THÔNG SỐ ĐÁNH GIÁ
# ==============================================================================
MAX_PROMPTS = 553 # @param {type:"integer"} # Số prompt cần đánh giá (553 chuẩn GenEval)
NUM_IO_WORKERS = 8 # @param {type:"integer"} # Số luồng I/O nạp song song ảnh vào RAM
SAVE_REPORT_TO_DRIVE = True # @param {type:"boolean"} # Lưu file báo cáo JSON và CSV lên Drive

# Tự động phân giải và tìm kiếm thư mục trên Drive hoặc SSD cục bộ
folder_queries = [q.strip() for q in TARGET_FOLDERS_INPUT.replace("\n", ",").split(",") if q.strip()]
resolved_targets = []

search_paths = [
    f"{DRIVE_DIR}/Target_samples",
    f"{WORKDIR}/Target_samples",
    "/content/drive/MyDrive/RS-LiDAR/Target_samples",
    "/content/drive/My Drive/RS-LiDAR/Target_samples"
]

for query in folder_queries:
    matched_in_search = False
    for sp in search_paths:
        pattern = os.path.join(sp, query)
        matches = glob.glob(pattern)
        if matches:
            for m in matches:
                real_m = os.path.realpath(m)
                existing_real = [os.path.realpath(x) for x in resolved_targets]
                if os.path.isdir(m) and real_m not in existing_real:
                    resolved_targets.append(m)
                    matched_in_search = True
            if matched_in_search:
                break
    if not matched_in_search:
        if os.path.exists(query) and os.path.isdir(query):
            real_q = os.path.realpath(query)
            existing_real = [os.path.realpath(x) for x in resolved_targets]
            if real_q not in existing_real:
                resolved_targets.append(query)

print("=" * 75)
print(f"📋 DANH SÁCH {len(resolved_targets)} THƯ MỤC HỢP LỆ ĐÃ XÁC NHẬN:")
for idx, t in enumerate(resolved_targets, 1):
    n_prompts_disk = len(glob.glob(f"{t}/[0-9]*/samples/*.png")) // 4 # ước tính
    print(f"  {idx:02d}. 📁 {os.path.basename(t)}")
    print(f"      Đường dẫn: {t}")
print("=" * 75)

if not resolved_targets:
    print("\n⚠️ [CẢNH BÁO]: Không tìm thấy thư mục nào khớp với input của bạn!")
    print(f"👉 Vui lòng kiểm tra lại tên thư mục trong {DRIVE_DIR}/Target_samples/")


## 5. Thực Thi Chấm Điểm Đa Dạng (In-Memory RAM Preloading & GPU Evaluation)
Bấm chạy cell dưới đây. Toàn bộ ảnh sẽ được nạp đa luồng vào RAM phiên làm việc và đo đạc trên GPU. Quá trình chạy cực nhanh và in ra bảng tổng hợp Markdown đối đầu trực diện.

In [ ]:
import os, json, importlib
import pandas as pd
from IPython.display import display, Markdown

# Tự động nạp lại module mới nhất (chống cache RAM Colab)
import compute_diversity_metrics
importlib.reload(compute_diversity_metrics)
from compute_diversity_metrics import evaluate_multiple_folders, format_markdown_table

assert len(resolved_targets) > 0, "❌ Không có thư mục hợp lệ để đánh giá. Vui lòng kiểm tra lại Mục 4."

# Đường dẫn lưu file báo cáo
out_json = f"{DRIVE_DIR}/diversity_summary_report.json" if SAVE_REPORT_TO_DRIVE else f"{WORKDIR}/diversity_summary_report.json"
out_csv = f"{DRIVE_DIR}/diversity_summary_report.csv" if SAVE_REPORT_TO_DRIVE else f"{WORKDIR}/diversity_summary_report.csv"

# Kích hoạt hệ thống đánh giá
summaries = evaluate_multiple_folders(
    folder_paths=resolved_targets,
    max_prompts=MAX_PROMPTS,
    max_workers=NUM_IO_WORKERS,
    output_json=out_json,
    output_csv=out_csv,
    verbose=True
)

# Hiển thị bảng Markdown tương tác đẹp mắt trên Colab
md_content = format_markdown_table(summaries)
display(Markdown("### 📊 BẢNG TỔNG HỢP SO SÁNH CHẤT LƯỢNG & ĐỘ ĐA DẠNG MẪU (TÍCH LŨY TOÀN BỘ SETTINGS):"))
display(Markdown(md_content))


## 6. Trực Quan Hóa Đường Biên Pareto (Quality vs Diversity Frontier)
Vẽ đồ thị phân tán 2 chiều trực diện thể hiện mối quan hệ giữa **Chất lượng (ImageReward)** và **Độ Đa Dạng (Vendi Score / Pairwise LPIPS)**.

In [ ]:
# @title 📈 Vẽ Biểu Đồ So Sánh Pareto (ImageReward vs Vendi Score / LPIPS)
import matplotlib.pyplot as plt
import numpy as np

valid_data = [s for s in summaries if "error" not in s and s.get("image_reward_max") is not None]

if len(valid_data) == 0:
    print("⚠️ Chưa có đủ dữ liệu có trường ImageReward để vẽ biểu đồ Pareto.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=150)
    
    # Palette màu sắc chuyên nghiệp chuẩn hội nghị
    colors = ['#ef4444', '#10b981', '#3b82f6', '#f59e0b', '#8b5cf6', '#ec4899', '#06b6d4', '#84cc16']
    
    def _get_scatter_label(item):
        r_name = item['run_name']
        m = item.get('method', 'RS-SMC' if 'RS' in r_name else ('SMC' if 'SMC' in r_name else r_name[:15]))
        n = item.get('particles', item.get('num_particles', 4))
        sig = item.get('sigma', '—')
        ann = item.get('anneal', '—')
        lbl = f"{m} (N={n})"
        if sig not in ('—', None, '', np.nan):
            lbl += f" σ={sig}"
        if ann == 'Có':
            lbl += " (Ann)"
        return lbl
    
    # --- SUBPLOT 1: ImageReward vs Vendi Score ---
    for idx, item in enumerate(valid_data):
        r_name = item['run_name']
        ir = item['image_reward_max']
        vs = item['vendi_score_mean']
        c = colors[idx % len(colors)]
        label = _get_scatter_label(item)
        
        ax1.scatter(vs, ir, color=c, s=180, edgecolors='black', linewidth=1.5, zorder=4, label=label)
        ax1.annotate(label, (vs, ir), textcoords="offset points", xytext=(8, 5), fontsize=10, weight='bold')
        
    ax1.set_title("Quality vs Semantic Modes (Pareto Frontier)", fontsize=13, weight='bold', pad=12)
    ax1.set_xlabel("Intra-Prompt Vendi Score (Effective Modes) ↑", fontsize=11, weight='bold')
    ax1.set_ylabel("ImageReward (Max) ↑", fontsize=11, weight='bold')
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(loc="lower right", framealpha=0.9)
    
    # --- SUBPLOT 2: ImageReward vs Pairwise LPIPS ---
    for idx, item in enumerate(valid_data):
        r_name = item['run_name']
        ir = item['image_reward_max']
        lp = item['lpips_mean']
        c = colors[idx % len(colors)]
        label = _get_scatter_label(item)
        
        ax2.scatter(lp, ir, color=c, s=180, edgecolors='black', linewidth=1.5, zorder=4, label=label)
        ax2.annotate(label, (lp, ir), textcoords="offset points", xytext=(8, 5), fontsize=10, weight='bold')
        
    ax2.set_title("Quality vs Visual Layout Diversity (Pareto Frontier)", fontsize=13, weight='bold', pad=12)
    ax2.set_xlabel("Intra-Prompt Pairwise LPIPS Distance ↑", fontsize=11, weight='bold')
    ax2.set_ylabel("ImageReward (Max) ↑", fontsize=11, weight='bold')
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend(loc="lower right", framealpha=0.9)
    
    plt.tight_layout()
    plot_save_path = f"{DRIVE_DIR}/pareto_quality_vs_diversity.png"
    plt.savefig(plot_save_path, bbox_inches='tight')
    print(f"📊 Đã xuất đồ thị Pareto chất lượng cao tại: {plot_save_path}")
    plt.show()


## 7. Trực Quan Hóa Đối Đầu Mắt Người (Human-Eye Visual Inspection Grid)
Xem trực tiếp $N$ hạt sinh ra của cùng một prompt giữa 2 thư mục để kiểm chứng hiện tượng trùng lặp hạt của Vanilla SMC so với sự phong phú đa dạng của RS-SMC.

In [ ]:
# @title 🖼️ So Sánh Lưới Ảnh N Hạt Của Cùng 1 Prompt Giữa Các Thư Mục
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

INSPECT_PROMPT_ID = 0 # @param {type:"integer"}
p_folder = f"{INSPECT_PROMPT_ID:05d}"

valid_dirs_to_show = [d for d in resolved_targets if os.path.exists(os.path.join(d, p_folder, "samples"))]

if not valid_dirs_to_show:
    print(f"⚠️ Không tìm thấy ảnh của prompt {p_folder} trong các thư mục đã chọn.")
else:
    n_rows = len(valid_dirs_to_show)
    # Lấy số hạt N của hàng đầu
    first_imgs = sorted(glob.glob(f"{valid_dirs_to_show[0]}/{p_folder}/samples/*.png"))
    n_cols = len(first_imgs)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3.5), dpi=120)
    if n_rows == 1:
        axes = [axes]
    if n_cols == 1:
        axes = [[ax] for ax in axes]
        
    for r_idx, r_dir in enumerate(valid_dirs_to_show):
        sample_files = sorted(glob.glob(f"{r_dir}/{p_folder}/samples/*.png"))
        short_title = os.path.basename(r_dir)[:35]
        for c_idx in range(n_cols):
            ax = axes[r_idx][c_idx] if n_rows > 1 else axes[0][c_idx]
            if c_idx < len(sample_files):
                img = Image.open(sample_files[c_idx])
                ax.imshow(img)
                ax.set_title(f"Hạt #{c_idx+1}", fontsize=10)
            ax.axis("off")
        # Label tên method ở hàng
        axes[r_idx][0].text(-0.1, 0.5, short_title, transform=axes[r_idx][0].transAxes,
                            fontsize=11, weight='bold', va='center', ha='right', rotation=90)
                            
    plt.suptitle(f"Đối Chiếu Đàn Hạt Tại Prompt #{p_folder}", fontsize=14, weight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
